In [1]:
##Importante tener cuenta en hugging face, para algunos modelos como meta/llama
# se debe tener aprobacion para descargar desde la plataforma el modelo. se debe ingresar a la pagina
#https://huggingface.co/google/gemma-3-1b-pt y aceptar los terminos del modelo para poderlo descargar con la cuenta de hugginface
from huggingface_hub import login
login(new_session=False)

In [ ]:
import os, torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel, PeftConfig
NB_DIR = Path().resolve()
ROOT_DIR = (NB_DIR / ".." / "..").resolve()

ADAPTER_DIR = Path(ROOT_DIR) / "modelos" / "gemma-3-1b-pt"   #carpeta local del LoRA/adaptador
DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float32
HF_TOKEN = os.getenv("HF_TOKEN")

# Descubre el base exacto que espera el adaptador
base_id = PeftConfig.from_pretrained(ADAPTER_DIR).base_model_name_or_path
print("Base esperado por el LoRA:", base_id)

# Si la página indica 'google/gemma-3-1b-pt' y coincide, úsalo; si tu PEFT dice otro, respeta el de PEFT.
BASE_ID = base_id or "google/gemma-3-1b-pt"

tokenizer = AutoTokenizer.from_pretrained(BASE_ID, use_fast=True, token=HF_TOKEN, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(BASE_ID, torch_dtype=DTYPE, device_map="auto",
                                             token=HF_TOKEN, trust_remote_code=True)
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()


Base esperado por el LoRA: google/gemma-3-1b-pt


`torch_dtype` is deprecated! Use `dtype` instead!


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): Gemma3ForCausalLM(
      (model): Gemma3TextModel(
        (embed_tokens): Gemma3TextScaledWordEmbedding(262144, 1152, padding_idx=0)
        (layers): ModuleList(
          (0-25): 26 x Gemma3DecoderLayer(
            (self_attn): Gemma3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1152, out_features=1024, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1152, out_features=4, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=4, out_features=1024, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
          

In [2]:
# ============================
# Generar PLS con CoT Implicito Qwen (HF) usando columnas: name, article, summary
# ============================

import re
import time  # NEW
from pathlib import Path
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm.auto import tqdm

@torch.inference_mode()
def generate_pls_batch(
    data: str,
    prompt_fn,
    batch_size: int = 2,
    max_new_tokens: int = 380,     # suficiente para 4–6 oraciones
    temperature: float = 0.0,      # determinista → mejor factualidad
    top_p: float = 1.0,
    num_beams: int = 1,
    repetition_penalty: float = 1.02,
    no_repeat_ngram_size: int = 4,
):
    texts = data['article'].fillna("").astype(str).tolist()
    df_out = data.copy()

    # Calcula un input máximo seguro
    max_ctx = getattr(model.config, "max_position_embeddings", 4096)
    max_input_len = max(8, max_ctx - max_new_tokens)

    outputs = []
    latencies = []  # NEW
    pbar = tqdm(total=len(texts), desc='Generando resúmenes', unit="sample")

    try:
        for i in range(0, len(texts), batch_size):
            batch_texts = texts[i:i + batch_size]
            prompts = [prompt_fn(t) for t in batch_texts]

            enc = tokenizer(
                prompts,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=max_input_len,
            )
            input_ids = enc["input_ids"].to(model.device)
            attention_mask = enc["attention_mask"].to(model.device)

            # --- NEW: medir tiempo del batch con sync de GPU
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t0 = time.perf_counter()

            gen_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=max_new_tokens,
                do_sample=(temperature > 0.0 or top_p < 1.0),
                temperature=temperature,
                top_p=top_p,
                num_beams=num_beams,
                no_repeat_ngram_size=no_repeat_ngram_size,
                repetition_penalty=repetition_penalty,
                use_cache=True,
                eos_token_id=tokenizer.eos_token_id,
                pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id,
            )

            if torch.cuda.is_available():
                torch.cuda.synchronize()
            t1 = time.perf_counter()
            per_item_latency = (t1 - t0)  # NEW

            # Cortar por muestra usando la longitud REAL (no el ancho padded)
            batch_out = []
            input_lens = attention_mask.sum(dim=1)  # [B]
            for row in range(input_ids.size(0)):
                ilen = int(input_lens[row].item())
                gen_only = gen_ids[row, ilen:]        # ← clave: corta desde fin del prompt de ESA fila
                text = tokenizer.decode(gen_only, skip_special_tokens=True).strip()
                batch_out.append(text)

            outputs.extend(batch_out)
            latencies.extend([per_item_latency] * len(batch_texts))  # NEW
            pbar.update(len(batch_texts))
    finally:
        pbar.close()

    df_out['gen_summary'] = outputs
    df_out['latency_s'] = latencies  # NEW
    return df_out

# O el CoT factual corto
def generar_prompt_cot(scientific_text: str) -> str:
    return f"""You are a helpful medical writer.
            Think briefly before answering:
            - Use only statements explicitly present in the source
            - Keep names and numbers exactly as written.
            - 4–6 sentences, ≤120 words. Do not show your reasoning
            Scientific text:
            {scientific_text}
            Plain summary:"""

DATA_DIR = Path("data-sources/pre-processed")
RESULTS_DIR = Path("models/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

df_test = pd.read_csv(DATA_DIR / "data_finetuning_test.csv")
df_output = generate_pls_batch(
    data=df_test,
    prompt_fn=generar_prompt_cot, 
    batch_size=4,
    max_new_tokens=380,
    temperature=0.0,
    top_p=1.0,
)

csv_out = RESULTS_DIR / "summaries_gemma3_COT.csv"
df_output.to_csv(csv_out, index=False, encoding="utf-8")
print(f"Guardado CSV con PLS: {csv_out}")
print("Latencia promedio (s):", df_output["latency_s"].mean())

#2.7 a 4.1 GB VRAM



Generando resúmenes:   0%|          | 0/380 [00:00<?, ?sample/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Guardado CSV con PLS: models\results\summaries_gemma3_COT.csv
Latencia promedio (s): 21.074454307368548
